In [1]:
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from openai import OpenAI
import httpx
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field          # Field from pydantic, NOT dataclasses
from typing import Literal, TypedDict, List, Annotated
import uuid
import base64
import ssl
import numpy as np
import json
from langgraph.graph import StateGraph, START, END
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.embeddings import Embeddings
from langchain_core.documents import Document

load_dotenv()
ssl_cert_file = "C:\\Users\\Ujjwal_Manna\\Downloads\\ca-certs.crt"
os.environ['REQUESTS_CA_BUNDLE'] = ssl_cert_file
os.environ['SSL_CERT_FILE'] = ssl_cert_file

client = httpx.Client(verify=ssl.create_default_context(cafile=ssl_cert_file))
client_id     = os.environ.get('LLM_CLIENT_ID')
client_secret = os.environ.get('LLM_CLIENT_SECRET')
auth_header   = f"Basic {base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode()}"

llm_openai = ChatOpenAI(
    model="gpt-oss-120b",
    base_url=os.environ.get("LLM_URL"),
    http_client=client,
    api_key="dummy value",
    default_headers={
        'accept': '*/*',
        'Content-Type': 'application/json',
        'Authorization': auth_header
    },
    temperature=0.0,
    streaming=True
)

# ── Embedding model ────────────────────────────────────────────────────────────
class DSXEmbeddingInterface(Embeddings):
    def __init__(self, url, model_name, http_client, default_headers, batch_size=32):
        self.client = OpenAI(
            base_url=url, http_client=http_client,
            api_key='not-empty', default_headers=default_headers
        )
        self.model_name = model_name
        self.batch_size = batch_size

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        all_embeddings = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i + self.batch_size]
            resp  = self.client.embeddings.create(model=self.model_name, input=batch)
            all_embeddings.extend([a.embedding for a in resp.data])
        return all_embeddings

    def embed_query(self, text: str) -> List[float]:
        return self.embed_documents([text])[0]

embedding_model = DSXEmbeddingInterface(
    url="",
    model_name="",
    http_client=httpx.Client(verify=ssl.create_default_context(cafile=ssl_cert_file)),
    default_headers={
        "x-correlation-id": str(uuid.uuid4()),
        'accept': '*/*', 'Content-Type': 'application/json',
        'Authorization': auth_header
    },
    batch_size=32
)

# ── Vector store ───────────────────────────────────────────────────────────────
class SimpleVectorStore:
    def __init__(self, documents=None, embedding_model=None, persist_directory=None):
        self.documents        = documents if documents is not None else []
        self.embedding_model  = embedding_model
        self.embeddings       = None
        self.persist_directory = persist_directory

    def add_documents(self):
        texts = [doc.page_content for doc in self.documents]
        new_embs = np.array(self.embedding_model.embed_documents(texts))
        self.embeddings = new_embs if self.embeddings is None else np.vstack([self.embeddings, new_embs])

    def add_documents_to_existing(self, new_documents):
        texts    = [doc.page_content for doc in new_documents]
        new_embs = np.array(self.embedding_model.embed_documents(texts))
        self.documents.extend(new_documents)
        self.embeddings = new_embs if self.embeddings is None else np.vstack([self.embeddings, new_embs])

    def similarity_search(self, query, k=3):
        q_emb  = np.array(self.embedding_model.embed_query(query))
        q_norm = np.linalg.norm(q_emb)
        d_norms = np.linalg.norm(self.embeddings, axis=1)
        sims    = np.dot(self.embeddings, q_emb) / (d_norms * q_norm)
        return [self.documents[i] for i in np.argsort(sims)[::-1][:k]]

    def mmr_search(self, query: str, k: int = 5, lambda_param: float = 0.5):
        """
        Maximum Marginal Relevance — balances relevance vs diversity.

        Each iteration picks the candidate doc that maximises:
            MMR(d) = λ × sim(d, query) − (1−λ) × max_{s in selected} sim(d, s)
                     ↑ relevance reward        ↑ redundancy penalty

        lambda_param = 1.0  →  pure similarity  (same as similarity_search)
        lambda_param = 0.0  →  pure diversity   (ignores query, bad)
        lambda_param = 0.5  →  balanced default
        """
        if self.embeddings is None or len(self.documents) == 0:
            return []

        k = min(k, len(self.documents))

        # Step 1 — embed the query and compute cosine sim to ALL docs at once
        q_emb   = np.array(self.embedding_model.embed_query(query))
        q_norm  = np.linalg.norm(q_emb) or 1e-10
        d_norms = np.linalg.norm(self.embeddings, axis=1)
        safe_d  = np.where(d_norms == 0, 1e-10, d_norms)

        # query_sims[i] = cosine similarity between doc i and the query  shape:(N,)
        query_sims = np.dot(self.embeddings, q_emb) / (safe_d * q_norm)

        # Step 2 — greedy MMR loop
        selected   = []          # chosen doc indices (in selection order)
        candidates = list(range(len(self.documents)))

        for step in range(k):
            if not candidates:
                break

            if step == 0:
                # No selected docs yet → no redundancy penalty → pick most similar
                best = max(candidates, key=lambda i: query_sims[i])
            else:
                # Embeddings of already-selected docs  shape:(n_sel, dim)
                sel_embs  = self.embeddings[selected]
                sel_norms = np.linalg.norm(sel_embs, axis=1)
                safe_sel  = np.where(sel_norms == 0, 1e-10, sel_norms)

                best_score = -float("inf")
                best       = None

                for idx in candidates:
                    relevance = query_sims[idx]

                    # Cosine similarity to EVERY already-selected doc  shape:(n_sel,)
                    sims_to_sel = np.dot(sel_embs, self.embeddings[idx]) / (safe_sel * safe_d[idx])
                    # Redundancy = worst-case: max similarity to any selected doc
                    redundancy  = float(np.max(sims_to_sel))

                    mmr_score = lambda_param * relevance - (1 - lambda_param) * redundancy
                    if mmr_score > best_score:
                        best_score = mmr_score
                        best       = idx

            selected.append(best)
            candidates.remove(best)

        return [self.documents[i] for i in selected]

    def save(self, directory=None):
        save_dir = directory or self.persist_directory
        if not save_dir:
            raise ValueError("No persist directory specified")
        os.makedirs(save_dir, exist_ok=True)
        np.save(os.path.join(save_dir, "embeddings.npy"), self.embeddings)
        docs_data = [{"page_content": d.page_content, "metadata": d.metadata} for d in self.documents]
        with open(os.path.join(save_dir, "documents.json"), "w") as f:
            json.dump(docs_data, f)
        print(f"Vector store saved to {save_dir}")

    @classmethod
    def load(cls, directory, embedding_model):
        embeddings = np.load(os.path.join(directory, "embeddings.npy"))
        with open(os.path.join(directory, "documents.json")) as f:
            docs_data = json.load(f)
        documents = [Document(page_content=d["page_content"], metadata=d["metadata"]) for d in docs_data]
        vs = cls(documents=documents, embedding_model=embedding_model, persist_directory=directory)
        vs.embeddings = embeddings
        print(f"Vector store loaded from {directory} ({len(documents)} docs)")
        return vs

# ── Load / build vector store ──────────────────────────────────────────────────
persist_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "vector_store"))

if os.path.exists(persist_dir) and os.path.exists(os.path.join(persist_dir, "embeddings.npy")):
    vectorstore = SimpleVectorStore.load(persist_dir, embedding_model)
else:
    loader      = PyPDFLoader("../../Docs/fabric-admin.pdf")
    new_docs    = loader.load()
    vectorstore = SimpleVectorStore(new_docs, embedding_model, persist_directory=persist_dir)
    vectorstore.add_documents()
    vectorstore.save()

# ── Part A: Naive vs MMR comparison ───────────────────────────────────────────
query = "How do I configure access permissions in OneLake?"

print("=== NAIVE similarity_search (k=3) ===")
for i, doc in enumerate(vectorstore.similarity_search(query, k=3)):
    print(f"  [{i+1}] page={doc.metadata.get('page','?')}  {doc.page_content[:130].strip()}\n")

print("=== MMR mmr_search (k=3, λ=0.5) ===")
for i, doc in enumerate(vectorstore.mmr_search(query, k=3, lambda_param=0.5)):
    print(f"  [{i+1}] page={doc.metadata.get('page','?')}  {doc.page_content[:130].strip()}\n")

print("Observation: MMR results should come from different pages/subtopics.")
print("If pages are identical across both searches, the λ is too high — try λ=0.3.")


C:\Users\Ujjwal_Manna\AppData\Local\Temp\ipykernel_12620\3689912252.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Vector store loaded from C:\projects\personal\langchn_practise\vector_store (580 docs)
=== NAIVE similarity_search (k=3) ===
  [1] page=183  Setting name Description
With this setting on, user access token information, including names and
emails, is sent to data sources

  [2] page=183  Setting name Description
With this setting on, user access token information, including names and
emails, is sent to data sources

  [3] page=289  Feedback
Was this page helpful?
Provide product feedback | Ask the community
OneLake tenant settings
Article • 11/21/2023
OneLake

=== MMR mmr_search (k=3, λ=0.5) ===
  [1] page=183  Setting name Description
With this setting on, user access token information, including names and
emails, is sent to data sources

  [2] page=56  Admin overview

  [3] page=154  2. In the Admin portal, select Capacity settings.
3. Select the capacity you want to configure notifications for.

Observation: MMR results should come from different pages/subtopics.
If pages are identic

In [2]:

# ══════════════════════════════════════════════════════════════════════════════
# Part B — Multi-Query RAG Graph
# Flow: rewrite_queries → retrieve → grade_docs → answer
# ══════════════════════════════════════════════════════════════════════════════

# ── State ─────────────────────────────────────────────────────────────────────
class RAGState(TypedDict):
    question:         str
    query_variations: List[str]   # original + 3 LLM-generated rephrasings
    retrieved_docs:   List        # union of all retrieval results (deduplicated)
    relevant_docs:    List        # subset that passed the relevance grader
    answer:           str         # final LLM response

# ── Node 1: Rewrite queries ────────────────────────────────────────────────────
# Generates 3 semantically different phrasings so we cast a wider retrieval net.
# The original question is always kept to avoid drift.
class QueryVariations(BaseModel):
    queries: List[str] = Field(
        description="Exactly 3 semantically different phrasings of the question. "
                    "Each phrasing must use different vocabulary and approach the "
                    "topic from a different angle."
    )

def rewrite_queries_node(state: RAGState) -> dict:
    question = state.get("question", "")
    if not question:
        return {}

    structured_llm = llm_openai.with_structured_output(QueryVariations)
    result: QueryVariations = structured_llm.invoke([
        SystemMessage(content=(
            "You are an expert at reformulating search queries. "
            "Given a question, generate exactly 3 alternative phrasings that: "
            "1) Use different vocabulary and terminology "
            "2) Approach the topic from different angles "
            "3) Would retrieve different but relevant documents. "
            "Do NOT just shuffle words — genuinely rephrase."
        )),
        HumanMessage(content=question)
    ])

    # Include original question so we never lose the user's intent
    all_queries = [question] + result.queries
    print(f"[rewrite] Generated {len(all_queries)} queries:")
    for q in all_queries:
        print(f"  • {q}")
    return {"query_variations": all_queries}

# ── Node 2: Retrieve & deduplicate ────────────────────────────────────────────
# Runs similarity_search for every query variation.
# Deduplicates by exact page_content — same chunk from two queries counted once.
# WHY multi-query improves recall:
#   Different phrasings hit different documents due to vocabulary gaps in embeddings.
#   Union of results covers broader semantic territory than any single query.
# TRADE-OFF:
#   More queries → higher recall (find more relevant docs)
#   But also higher chance of including irrelevant docs → grade_docs_node fixes this
def retrieve_node(state: RAGState) -> dict:
    all_docs      = []
    seen_contents = set()

    for query in state["query_variations"]:
        docs = vectorstore.similarity_search(query, k=3)
        for doc in docs:
            if doc.page_content not in seen_contents:
                all_docs.append(doc)
                seen_contents.add(doc.page_content)

    print(f"[retrieve] {len(all_docs)} unique docs from {len(state['query_variations'])} queries")
    return {"retrieved_docs": all_docs}

# ── Node 3: Grade relevance ───────────────────────────────────────────────────
# LLM judges each retrieved doc: is it actually useful for answering the question?
# This corrects the recall vs precision trade-off from multi-query retrieval.
# NOTE: One LLM call per document. Optimisation Q answer:
#   → Batch all docs into a single prompt instead of N separate calls.
#     "Given the question, rate each of these N docs as relevant/not-relevant."
#     Returns a List[bool]. Saves N-1 LLM round-trips.
class DocGrade(BaseModel):
    relevant: bool = Field(description="True if the document helps answer the question")
    reason:   str  = Field(description="One sentence explaining the relevance decision")

def grade_docs_node(state: RAGState) -> dict:
    question      = state["question"]
    docs          = state["retrieved_docs"]
    grader        = llm_openai.with_structured_output(DocGrade)
    relevant_docs = []

    for doc in docs:
        grade: DocGrade = grader.invoke([
            SystemMessage(content=(
                "You are a strict document relevance grader. "
                "Given a question and a document excerpt, decide if the document "
                "directly helps answer the question. "
                "Be strict: only mark relevant=True if the document contains "
                "information that is specifically useful for the question."
            )),
            HumanMessage(content=(
                f"Question: {question}\n\n"
                f"Document (page {doc.metadata.get('page','?')}):\n"
                f"{doc.page_content[:600]}"
            ))
        ])
        status = "✓ RELEVANT" if grade.relevant else "✗ rejected"
        print(f"  [{status}] page={doc.metadata.get('page','?')} — {grade.reason[:80]}")
        if grade.relevant:
            relevant_docs.append(doc)

    print(f"[grade] {len(relevant_docs)}/{len(docs)} docs kept after grading")
    return {"relevant_docs": relevant_docs}

# ── Node 4: Generate answer ────────────────────────────────────────────────────
# Receives only the graded-relevant docs.
# Builds a context block with page citations, then prompts the LLM.
def answer_node(state: RAGState) -> dict:
    question      = state["question"]
    relevant_docs = state["relevant_docs"]

    if not relevant_docs:
        return {"answer": (
            "I could not find relevant information in the knowledge base "
            "to answer this question."
        )}

    # Build numbered context with page citations
    context_parts = [
        f"[Doc {i+1} — page {doc.metadata.get('page','?')}]\n{doc.page_content}"
        for i, doc in enumerate(relevant_docs)
    ]
    context = "\n\n---\n\n".join(context_parts)

    response: AIMessage = llm_openai.invoke([
        SystemMessage(content=(
            "You are a helpful assistant that answers questions strictly from "
            "the provided document context. "
            "Cite the Doc number and page when referencing specific information. "
            "If the context is insufficient, say so clearly."
        )),
        HumanMessage(content=f"Question: {question}\n\nContext:\n{context}")
    ])

    return {"answer": response.content}

# ── Graph construction ─────────────────────────────────────────────────────────
graph = StateGraph(RAGState)
graph.add_node("rewrite_queries", rewrite_queries_node)
graph.add_node("retrieve",        retrieve_node)
graph.add_node("grade_docs",      grade_docs_node)
graph.add_node("answer",          answer_node)

graph.add_edge(START,             "rewrite_queries")
graph.add_edge("rewrite_queries", "retrieve")
graph.add_edge("retrieve",        "grade_docs")
graph.add_edge("grade_docs",      "answer")
graph.add_edge("answer",          END)

rag_app = graph.compile()
print(rag_app.get_graph().draw_mermaid())


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	rewrite_queries(rewrite_queries)
	retrieve(retrieve)
	grade_docs(grade_docs)
	answer(answer)
	__end__([<p>__end__</p>]):::last
	__start__ --> rewrite_queries;
	grade_docs --> answer;
	retrieve --> grade_docs;
	rewrite_queries --> retrieve;
	answer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pyppeteer\us_keyboard_layout.py:73: SyntaxWarning: invalid escape sequence '\('
  'Digit9': {'keyCode': 57, 'code': 'Digit9', 'shiftKey': '\(', 'key': '9'},
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pyppeteer\us_keyboard_layout.py:143: SyntaxWarning: invalid escape sequence '\<'
  'Comma': {'keyCode': 188, 'code': 'Comma', 'shiftKey': '\<', 'key': ','},
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pyppeteer\us_keyboard_layout.py:247: SyntaxWarning: invalid escape sequence '\<'
  '<': {'keyCode': 188, 'key': '\<', 'code': 'Comma'},


In [3]:

# ── Run the full pipeline ──────────────────────────────────────────────────────
result = rag_app.invoke({
    "question":         "What security and access control options are available in Microsoft Fabric OneLake?",
    "query_variations": [],
    "retrieved_docs":   [],
    "relevant_docs":    [],
    "answer":           ""
})

print("\n" + "═" * 60)
print(f"Query variations  : {len(result['query_variations'])}")
print(f"Docs retrieved    : {len(result['retrieved_docs'])}")
print(f"Docs after grading: {len(result['relevant_docs'])}")
print("═" * 60)
print("\nFINAL ANSWER:\n")
print(result["answer"])


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QueryVariations(queries=[...oft Fabric ecosystem?']), input_type=QueryVariations])
  return self.__pydantic_serializer__.to_python(


[rewrite] Generated 4 queries:
  • What security and access control options are available in Microsoft Fabric OneLake?
  • Which authentication and permission mechanisms does OneLake in Microsoft Fabric support?
  • How can I manage data protection and user rights within Microsoft Fabric's OneLake storage?
  • What governance and access‑management features are available for OneLake in the Microsoft Fabric ecosystem?
[retrieve] 4 unique docs from 4 queries


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=DocGrade(relevant=False, ...y answer the question.'), input_type=DocGrade])
  return self.__pydantic_serializer__.to_python(


  [✗ rejected] page=15 — The excerpt only provides a high-level overview of Microsoft Fabric security and


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=DocGrade(relevant=True, r...nswering the question.'), input_type=DocGrade])
  return self.__pydantic_serializer__.to_python(


  [✓ RELEVANT] page=183 — The excerpt lists specific security and access control settings for OneLake, suc


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=DocGrade(relevant=False, ...l options for OneLake.'), input_type=DocGrade])
  return self.__pydantic_serializer__.to_python(


  [✗ rejected] page=13 — The excerpt provides a generic overview of Fabric governance and security topics


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=DocGrade(relevant=False, ...l options for OneLake.'), input_type=DocGrade])
  return self.__pydantic_serializer__.to_python(


  [✗ rejected] page=289 — The excerpt only mentions tenant settings and general access via external and in
[grade] 1/4 docs kept after grading

════════════════════════════════════════════════════════════
Query variations  : 4
Docs retrieved    : 4
Docs after grading: 1
════════════════════════════════════════════════════════════

FINAL ANSWER:

Microsoft Fabric’s OneLake includes a set of built‑in security and access‑control settings that let administrators decide how users authenticate, what data they can see, and how tightly connections are protected. According to the settings table on page 183 of the documentation, the available options are:

| Security / Access‑Control option | What it does |
|--------------------------------|--------------|
| **Send user access‑token information to on‑premises data gateway** – When enabled, the user’s name and email are passed to the data source via the gateway for authentication. |
| **Allow users to view Power BI files saved in OneDrive and Sha